#  VN Food VQA — Generate QA Pairs

| Cell | Nội dung |
|------|----------|
| **#0** | Cài đặt thư viện |
| **#1** | Import |
| **#2** | Cấu hình tài khoản (API keys) |
| **#3** | Cấu hình logic xử lý (model, batch, part) |
| **#4** | PROMPT |
| **#5** | Helpers (resize ảnh, format dataset) |
| **#6** | Hàm gọi API (1 batch + prompt / request) |
| **#7** | Worker (chạy song song nhiều key) |
| **#8** | Chia ảnh → batch → part |
| **#9** | Gán part cho từng API key |
| **#10** | Demo: 1 batch, 1 tài khoản |
| **#11** | Full: chạy song song nhiều key |

---
### Tính toán capacity
```
Ưu tiên model (chất lượng cao → thấp):
  gemini-3-flash       → 20 req/ngày
  gemini-3.1-flash     → 20 req/ngày  (nếu available)
  gemini-2.5-flash     → 20 req/ngày
  gemini-2.5-flash-lite → 20 req/ngày

1 tài khoản × 4 model × 20 req = 80 req × 15 ảnh = 1200 ảnh/tài khoản
10 tài khoản × 2 model tốt nhất × 20 req × 15 ảnh = 6000 ảnh
```

In [ ]:
# ============================================================
# Cell #0 — Cài đặt thư viện
# ============================================================
!pip install -q google-genai

In [ ]:
# ============================================================
# Cell #1 — Import
# ============================================================
import os, re, json, time, math, threading, glob
import pandas as pd
from io import BytesIO
from datetime import datetime
from PIL import Image
from google import genai
from google.genai import types
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive, userdata

print('✅ Cell #1 Import xong.')

✅ Cell #1 Import xong.


In [ ]:
# ============================================================
# Cell #2 — Cấu hình tài khoản
# Mỗi tài khoản = 1 dict {secret_name, models[]}
# models: danh sách model ưu tiên cho tài khoản đó
# ============================================================

# Thứ tự ưu tiên model (chất lượng cao → thấp)
MODEL_PRIORITY = [
    'gemini-2.5-flash',       # tốt nhất hiện tại
    'gemini-2.5-flash-lite'   # fallback
]

# Cấu hình từng tài khoản
ACCOUNT_CONFIGS = [
    {'secret': 'vanh1', 'models': ['gemini-2.5-flash', 'gemini-2.5-flash-lite']},
    {'secret': 'vanh2', 'models': ['gemini-2.5-flash', 'gemini-2.5-flash-lite']},
    {'secret': 'vanh3', 'models': ['gemini-2.5-flash', 'gemini-2.5-flash-lite']},
    {'secret': 'vanh4', 'models': ['gemini-2.5-flash', 'gemini-2.5-flash-lite']},
    {'secret': 'vanh5', 'models': ['gemini-2.5-flash', 'gemini-2.5-flash-lite']},
    {'secret': 'vanh6', 'models': ['gemini-2.5-flash', 'gemini-2.5-flash-lite']},
    {'secret': 'vanh7', 'models': ['gemini-2.5-flash', 'gemini-2.5-flash-lite']},
    {'secret': 'vanh8', 'models': ['gemini-2.5-flash', 'gemini-2.5-flash-lite']},
    {'secret': 'vanh9', 'models': ['gemini-2.5-flash', 'gemini-2.5-flash-lite']},
    {'secret': 'vanh10', 'models': ['gemini-2.5-flash', 'gemini-2.5-flash-lite']}
    # Thêm tài khoản tại đây:
    # {'secret': 'Key_3', 'models': ['gemini-2.5-flash', 'gemini-2.5-flash-lite']},
]


# Load API keys từ Colab Secrets
ACCOUNTS = []
for cfg in ACCOUNT_CONFIGS:
    try:
        key = userdata.get(cfg['secret'])
        if key:
            ACCOUNTS.append({'key': key, 'models': cfg['models'], 'name': cfg['secret']})
            print(f'  ✅ {cfg["secret"]} loaded | models: {cfg["models"]}')
    except Exception:
        print(f'  ⚠️  {cfg["secret"]} không tìm thấy trong Secrets')

assert len(ACCOUNTS) > 0, 'Chưa có tài khoản nào hợp lệ!'
print(f'\nTổng tài khoản hợp lệ: {len(ACCOUNTS)}')

  ✅ vanh1 loaded | models: ['gemini-2.5-flash', 'gemini-2.5-flash-lite']
  ✅ vanh2 loaded | models: ['gemini-2.5-flash', 'gemini-2.5-flash-lite']
  ✅ vanh3 loaded | models: ['gemini-2.5-flash', 'gemini-2.5-flash-lite']
  ✅ vanh4 loaded | models: ['gemini-2.5-flash', 'gemini-2.5-flash-lite']
  ✅ vanh5 loaded | models: ['gemini-2.5-flash', 'gemini-2.5-flash-lite']
  ✅ vanh6 loaded | models: ['gemini-2.5-flash', 'gemini-2.5-flash-lite']
  ✅ vanh7 loaded | models: ['gemini-2.5-flash', 'gemini-2.5-flash-lite']
  ✅ vanh8 loaded | models: ['gemini-2.5-flash', 'gemini-2.5-flash-lite']
  ✅ vanh9 loaded | models: ['gemini-2.5-flash', 'gemini-2.5-flash-lite']
  ✅ vanh10 loaded | models: ['gemini-2.5-flash', 'gemini-2.5-flash-lite']

Tổng tài khoản hợp lệ: 10


In [ ]:
# ============================================================
# Cell #3 — Cấu hình logic xử lý
# ============================================================

# --- Chế độ chạy ---
MODE = 'full'  # 'demo' | 'full'

# --- Batch & Part ---
BATCH_IMAGES  = 15   # số ảnh / request (tối ưu: 15 để không bị cắt output)
MAX_REQ_MODEL = 20   # số request tối đa / model / tài khoản / ngày
# → 1 part = MAX_REQ_MODEL batch = 20 × 15 = 300 ảnh

# --- Rate limit ---
SLEEP_SEC   = 15    # giây nghỉ giữa 2 request (< 10 RPM)
BATCH_SLEEP = 30   # giây nghỉ sau mỗi part

# --- Đường dẫn ---
drive.mount('/content/drive')
BASE_DIR   = '/content/drive/MyDrive/Data_DL1'
CSV_PATH   = f'{BASE_DIR}/sampled_metadata_5000.csv'
IMAGE_DIR  = f'{BASE_DIR}/vietnamese_food_images'
OUTPUT_DIR = f'{BASE_DIR}/Output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Tính toán capacity ---
n_accounts     = len(ACCOUNTS)
n_models       = len(ACCOUNTS[0]['models'])  # giả sử tất cả cùng số model
imgs_per_acct  = MAX_REQ_MODEL * n_models * BATCH_IMAGES
total_capacity = imgs_per_acct * n_accounts

print(f'BATCH_IMAGES     : {BATCH_IMAGES} ảnh/request')
print(f'MAX_REQ_MODEL    : {MAX_REQ_MODEL} request/model/tài khoản')
print(f'Ảnh/tài khoản    : {MAX_REQ_MODEL} × {n_models} models × {BATCH_IMAGES} = {imgs_per_acct}')
print(f'Tổng capacity    : {imgs_per_acct} × {n_accounts} tài khoản = {total_capacity} ảnh')
print(f'\nMode : {MODE}')
print(f'Sleep: {SLEEP_SEC}s/request | {BATCH_SLEEP}s/part')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BATCH_IMAGES     : 15 ảnh/request
MAX_REQ_MODEL    : 20 request/model/tài khoản
Ảnh/tài khoản    : 20 × 2 models × 15 = 600
Tổng capacity    : 600 × 10 tài khoản = 6000 ảnh

Mode : full
Sleep: 15s/request | 30s/part


In [ ]:
# ============================================================
# Cell #4 — PROMPT
# Chỉ sửa cell này khi muốn thay đổi nội dung Q&A
# ============================================================

# PROMPT = """
# Quy chuẩn chung:

# Bước 1: Mô tả chi tiết

#     Liệt kê: Mọi vật thể (món ăn, dụng cụ, đồ trang trí) và tổng số vật thể có trong ảnh.

#     Đặc điểm: Chỉ rõ màu sắc, hình dạng, kích thước tương đối và số lượng của từng vật.

#     Vị trí: Vị trí tuyệt đối trong ảnh (trên/dưới/trái/phải...) và vị trí tương đối (so với các vật khác).

# Bước 2: Sinh Q&A theo 4 nhóm (Tuân thủ nghiêm ngặt số lượng) từ mô tả từ bước 1

#     Nhóm 1 - "0_Overall_" + {1/2/3/4/5} (Đúng 5 câu): 3 cách hỏi khác nhau về nội dung chính của ảnh. Đáp án là 5 câu trả lời khác nhau về tên món ăn (trả lời hoàn chỉnh 1 câu chứ không phải chỉ nhắc cụm từ).

#     Nhóm 2 - "1_Visual_yes_no_[number]": Câu hỏi Có/Không bao quát 5 khía cạnh: màu sắc, kích thước, hình dạng, số lượng, vị trí. Các câu trả lời: Có/ Không, mô tả đúng ở trên ảnh. Chứ không chỉ Có, Không.

#     Nhóm 3 - "1_Visual_open_end_[number]": Câu hỏi mở bao quát 3 khía cạnh (như Nhóm 2) cho từng vật thể.

#     Nhóm 4 - "2_ExternalKnowledge" + {1/2/3/4} (Đúng 4 câu): Hỏi về kiến thức bên ngoài (xuất xứ, nguyên liệu, cách thưởng thức).

# Lưu ý:
# - Ở Nhóm 2, 3: Số sau _ là index vật thể (number = 1 = vật thể 1, number = 2 = vật thể 2...) — sinh cho từng vật thể chính trong (tối đa 3 vật thể * 6 khía cạnh)

# Ví dụ bạn có thể áp dụng: Ví dụ với  1 ảnh có bàn ăn phở, tô phở, có quẩy và trà đá
#  Mô tả:
#  Có 4 vật thể trong ảnh là: bàn, cốc trà đá, dĩa quẩy, bát phở (các vật thể và số lượng vật thể)
#  Có 1 cái bàn, 1 cốc trà đá, 1 dĩa quẩy, 3 cái quẩy, 1 bát phở (số lượng của từng vật thể)
#  Bàn - màu xám, kích thước lớn, có hình vuông, vị trí ở giữa bức ảnh (chi tiết cho vật thể "Bàn")
#  Dĩa quẩy (vật thể chính) - đây là món ăn kèm với phở có màu cam nhạt ăn với nước phở rất béo (mô tả), dĩa có màu xanh, hình tròn, vị trí ở bên trái bức ảnh (chi tiết cho vật thể "dĩa quẩy")
#  Phở (vật thể chính) - mô tả về vật liệu trong tô (nhiều hành, phở bò/ phở gà, ...), tô phở có màu trắng, hình tròn, vị trí ở bên phải (chi thết cho vật thể "Phở")
#  Bàn nằm dưới dĩa quẩy, tô phở, cốc trà đá, Dĩa quầy so với tô phở, .... (vị trí tương đối so với nhau)

#  Từ mô tả bạn dựa thành các cặp câu hỏi và trả lời nhé:
#  Overivew:
#  (3 câu nhận dạng chung về món chính)
#  Có gì trong ảnh ? -> Ảnh trên có món phở bò
#  Bức ảnh mô tả về món ăn gì ? -> Bức ảnh có phở
#  Món ăn gì xuất hiện ở trong ảnh ? -> Có món phở ở trong ảnh
#  (1 câu về số lượng và 1 câu hỏi chi tiết về các vật thể)
#  Có bao nhiêu vật thể trong ảnh ? -> Có 4 vật thể trong ảnh
#  Các vật thể ở trong ảnh là ? Bức ảnh mô tả về bàn thức ăn có tô phở, dĩa quẩy và cốc trà đá.
#  câu hỏi yes/no và open-end dựa trên 6 khía cạnh và số lương vật thể (ở đây có 3 cái chính quẩy, phở, trà đá):
#  Quẩy là món ăn kèm với phở đúng không (nhận dạng) -> Đúng,đây là món ăn kèm với phở rất nổi tiếng ở miền Bắc.
#  Có 3 cái quẩy trong hình đúng không (số lượng) -> Có, trên dĩa có 5 cái quẩy
#  ... tương tự cho phở và cốc trà đá và cuối kì là 4 câu hỏi cho kiến thức ngoài nhé
#  Bạn lưu ý trả lời ở phần visual trong từng khía cạnh cụ thể là tên từng món nhé (không phải là món chính có màu trắng phải không -> vì mô tả bạn đã đã xác định món chính, vật thể phụ nên ở đây phải hỏi cụ thể Phở có bánh phở màu trắng đúng không ?)
# Số lượng riêng của các vật thể rất quan trọng nhé. Bạn hãy đếm cho thật kĩ (nếu ví dụ trên là số quẩy trong dĩa quẩy).

# - Không dùng JSON, chỉ text thuần theo đúng định dạng trên.

# Với TỪNG ảnh được đánh số, liệt kê theo đúng định dạng sau
# (mỗi Q&A trên 2 dòng liên tiếp, không thêm dòng trống giữa chúng):
# [image_id]
# overall_0: <câu hỏi>
# <câu trả lời>
# (3 cặp câu hỏi trả lời overall khác)
# ...
# yes_no_identify_num: <câu hỏi về sự tồn tại vật thể>
# <câu trả lời>
# (kèm với 4 câu hỏi trả lời về các đặc điểm khác)
# ...
# open_end_identify_num: <câu hỏi mô tả vật thể>
# <câu trả lời>
# ...
# (kèm với 4 câu hỏi trả lời về các đặc điểm khác)
# external_0: <câu hỏi xuất xứ>
# <câu trả lời>
# (kèm với 3 cặp câu hỏi trả lời về external khác)
# """

# print(f'✅ Cell #4 PROMPT loaded: {len(PROMPT)} ký tự (~{len(PROMPT)//4} tokens)')

In [ ]:
# ============================================================
# Cell #4 — PROMPT
# Chỉ sửa cell này khi muốn thay đổi nội dung Q&A
# ============================================================

PROMPT = """
Quy chuẩn chung:
Bước 1: Mô tả

    Liệt kê: Mọi vật thể (món ăn, dụng cụ, đồ trang trí) và tổng số vật thể có trong ảnh.

    Đặc điểm: Chỉ rõ màu sắc, hình dạng, kích thước tương đối và số lượng của từng vật.

    Vị trí: Vị trí tuyệt đối trong ảnh (trên/dưới/trái/phải...) và vị trí tương đối (so với các vật khác).

Bước 2: Sinh Q&A theo 4 nhóm (Tuân thủ nghiêm ngặt số lượng) từ mô tả từ bước 1

    Nhóm 1 - "Overall_" + {1/2/3/4} (Đúng 3 câu): 4 cách hỏi khác nhau về vật thể chính (món ăn chính) của ảnh. Đáp án là 4 câu trả lời khác nhau về tên món ăn (trả lời hoàn chỉnh 1 câu chứ không phải chỉ nhắc cụm từ). (3 câu hỏi về món chính, 1 câu hỏi về các vật thể, 1 câu hỏi về số lượng vật thể).

    Nhóm 2 - "Visual_yes_no_[number]": Câu hỏi Có/Không bao quát 4 khía cạnh: đặc điểm món, màu sắc, hình dạng, số lượng. Các câu trả lời: Có/ Không, mô tả đúng ở trên ảnh. Chứ không chỉ Có, Không.

    Nhóm 3 - "Visual_open_end_[number]": Câu hỏi mở bao quát 4 khía cạnh (như Nhóm 2) cho từng vật thể.

    Nhóm 4 - "ExternalKnowledge" + {1/2/3} (Đúng 3 câu): Hỏi về kiến thức bên ngoài (xuất xứ, nguyên liệu, cách thưởng thức).

Lưu ý:
- Ở Nhóm 2, 3: Số sau _ là index vật thể (number = 1 = vật thể 1, number = 2 = vật thể 2...) — sinh cho từng vật thể chính trong (tối đa 3 vật thể * 4 khía cạnh)

Bạn sẽ sinh câu hỏi trả lời thành từ dòng liên tục nhau (tiltle 1 /câu hỏi 1 /n trả lời1 / title 2/ câu hỏi2  /n trả lời 2 ...).
Ví dụ: Tôi có 1 bức ảnh về món Phở Hà Nội.

Bước 1: Mô tả chi tiết
Bức ảnh trên là về món Phở truyền thống ở Hà Nội. Bức ảnh bao gồm các vật thể chính là cái bàn, tô phở, dĩa quầy và cốc trà đá (đây là liệt kê). Trong bức ảnh trên có 1 tô phở bò ít hành, tô màu trắng, hình tròn, có 1 dĩa quầy có 5 cái quẩy, quẩy có màu cam, dài, có 1 cốc trà đá, màu vàng, có hình trụ (đây là đặc điểm). Cái bàn nằm ở dưới tô phở, cốc trà đá, dĩa quầy. Tô phở, cốc trà đá nằm ở bên phải so với dĩa quẩy. Dĩa quẩy, tô phở nằm ở phía bên phải so với tô phở. Tô phở nằm chính giữa dĩa quẩy và cốc trà đá (ví trị tương đối so với nhau). Tô phở nằm ở chính giữa bức ảnh, cái bàn nằm ở dưới bức ảnh, dĩa quẩy nằm ở góc trên bên phải, cốc trà đá nằm ở góc trên bên trái.

Bước 2: Đặt câu hỏi (ví dụ ảnh có image_id = 0)
Cấu trúc là:
imageid
title /n Question /n Answer

image_0
Overall_0 /n Có gì trong bức ảnh ? /n Bức ảnh có món ăn là Phở.
Overall_1 /n Bức ảnh có chứa món ăn gì ? Có món phở Hà Nội trong bức ảnh.
Overall_2 /n Bức ảnh trên mô tả về gì ? Bức ảnh trên mô tả về món ăn, cụ thể là món Phở Hà Nội.
(3 câu hỏi 3 cách diễn đạt khác nhau)
Overall_3 /n Có gì trong bức ảnh /n Có 2 vật thể chính trong bức ảnh: tô phở, dĩa quẩy (câu hỏi về vật thể và số lượng của chúng)
(chú ý chỉ liệt kê 2 cái chính)
Visual_yes_no_feature_1 (Tô Phở)(món chính)
Visual_yes_no_feature_2 (quẩy)(món ăn kèm, các vật thể khác không phải đồ ăn thì bỏ qua)
Visual_yes_no_color_1 (Tô Phở)
Visual_yes_no_color_2 (quẩy)
Visual_yes_no_shape_1 (Tô Phở)
Visual_yes_no_shape_2 (quẩy)
Visual_yes_no_couting_1 (Tô Phở)
Visual_yes_no_counting_2 (quẩy)
Visual_open_end_feature_1 (Tô Phở)
Visual_open_end_feature_2 (quẩy)
Visual_open_end_color_1 (Tô Phở)
Visual_open_end_color_2 (quẩy)
Visual_open_end_shape_1 (Tô Phở)
Visual_open_end_shape_2 (quẩy)
Visual_open_end_couting_1 (Tô Phở)
Visual_open_end_counting_2 (quẩy)
External_0
External_1
External_2
External_3

Tổng cộng có 24 câu hỏi cho 1 bức ảnh nhé. Tôi cung cấp cho bạn 1 danh sách các ảnh với từng ảnh được đánh số, liệt kê theo image_id
"""

print(f'✅ Cell #4 PROMPT loaded: {len(PROMPT)} ký tự (~{len(PROMPT)//4} tokens)')

✅ Cell #4 PROMPT loaded: 3486 ký tự (~871 tokens)


In [ ]:
# ============================================================
# Cell #5 — Helpers
# ============================================================

def image_to_bytes(image_path, max_side=512):
    """Resize ảnh về tối đa 512px → bytes JPEG."""
    buf = BytesIO()
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    if max(w, h) > max_side:
        scale = max_side / max(w, h)
        img   = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    img.save(buf, format='JPEG', quality=85)
    return buf.getvalue()

# def parse_text_response(raw_text, image_ids):
#     """
#     Parse text thuần model trả về → list dict.
#     Model trả về 'image_N' ở đầu dòng (không có dấu []).
#     image_ids: list id thực tế tương ứng image_1, image_2, ...
#     """
#     blocks = re.split(r'(?m)^image_(\d+)\s*$', raw_text.strip())
#     pairs  = [(blocks[i], blocks[i+1])
#                for i in range(1, len(blocks)-1, 2)]
#     results = []
#     for img_num_str, content in pairs:
#         img_num = int(img_num_str) - 1
#         if img_num >= len(image_ids):
#             continue
#         image_id = image_ids[img_num]

#         qna   = []
#         lines = [l.strip() for l in content.strip().splitlines()
#                  if l.strip()]
#         i = 0
#         while i < len(lines) - 1:
#             m = re.match(
#                 r'^(overall|yes_no_\w+|open_end_\w+|external)_(\d+):\s*(.+)$',
#                 lines[i]
#             )
#             if m:
#                 raw_cat  = m.group(1)
#                 obj_idx  = m.group(2)
#                 question = m.group(3).strip()
#                 answer   = lines[i+1].strip()

#                 # Map category & question_type (bỏ prefix 0_, 1_, 2_)
#                 if raw_cat == 'overall':
#                     category      = '0_Overall'
#                     question_type = 'Overall'
#                 elif raw_cat.startswith('yes_no_'):
#                     attr          = raw_cat.replace('yes_no_', '')
#                     category      = f'1_Visual_yes_no_{attr}'
#                     question_type = f'yes_no_{attr}'
#                 elif raw_cat.startswith('open_end_'):
#                     attr          = raw_cat.replace('open_end_', '')
#                     category      = f'1_Visual_open_end_{attr}'
#                     question_type = f'open_end_{attr}'
#                 else:
#                     category      = '2_ExternalKnowledge'
#                     question_type = 'ExternalKnowledge'

#                 # sample_id = image_id + question_type + obj_idx
#                 sample_id = f'{image_id}_{question_type}_{obj_idx}'

#                 qna.append({
#                     'category'     : category,
#                     'question_type': question_type,
#                     'question'     : question,
#                     'answer'       : answer,
#                     'sample_id'    : sample_id
#                 })
#                 i += 2
#             else:
#                 i += 1

#         results.append({
#             'image_id'  : str(image_id),
#             'image_path': f'images/image_{image_id}.jpg',
#             'qna'       : qna
#         })
#     return results

def format_final_dataset(parsed_results, dataset_name='VN_Food_VQA'):
    """Chuyển list dict → dataset phẳng chuẩn."""
    data          = []
    unique_images = set()
    for item in parsed_results:
        image_id = item['image_id']
        unique_images.add(image_id)
        for qa in item['qna']:
            data.append({
                'image_id'     : image_id,
                'image_path'   : item['image_path'],
                'question_type': qa['question_type'],
                'question'     : qa['question'],
                'answer'       : qa['answer'],
                'sample_id'    : qa['sample_id']
            })
    return {
        'metadata': {
            'dataset'      : dataset_name,
            'created_at'   : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'total_images' : len(unique_images),
            'total_QAs'    : len(data)
        },
        'data': data
    }

print('✅ Cell #5 Helpers loaded.')

✅ Cell #5 Helpers loaded.


In [ ]:
def parse_text_response(raw_text, image_ids):
    """
    Parse text model trả về theo PROMPT mới.
    Format mỗi dòng: Title /n Question /n Answer
    Ví dụ: Overall_0 /n Có gì trong ảnh? /n Bức ảnh có món phở.
    """
    # Tách thành block theo image_N
    blocks = re.split(r'(?m)^image_(\d+)\s*$', raw_text.strip())
    pairs  = [(blocks[i], blocks[i+1])
               for i in range(1, len(blocks)-1, 2)]

    results = []
    for img_num_str, content in pairs:
        img_num = int(img_num_str) - 1
        if img_num >= len(image_ids):
            continue
        image_id = str(image_ids[img_num])

        qna   = []
        lines = [l.strip() for l in content.strip().splitlines() if l.strip()]

        for line in lines:
            # Format mới: "Title /n Question /n Answer"
            # hoặc "Title /n Question" (answer dòng tiếp theo)
            if ' /n ' not in line:
                continue

            parts = [p.strip() for p in line.split(' /n ')]
            if len(parts) < 2:
                continue

            title    = parts[0]   # Overall_0, Visual_yes_no_feature_1, ...
            question = parts[1] if len(parts) > 1 else ''
            answer   = parts[2] if len(parts) > 2 else ''

            # Map title → category + question_type
            title_lower = title.lower()

            if title_lower.startswith('overall'):
                # Overall_0, Overall_1, ...
                idx           = re.search(r'\d+', title)
                obj_idx       = idx.group() if idx else '0'
                category      = '0_Overall'
                question_type = 'Overall'

            elif 'yes_no' in title_lower:
                # Visual_yes_no_feature_1, Visual_yes_no_color_2, ...
                m = re.search(
                    r'yes_no_(\w+?)_(\d+)$', title, re.IGNORECASE
                )
                if not m:
                    continue
                attr          = m.group(1)
                obj_idx       = m.group(2)
                category      = f'1_Visual_yes_no_{attr}'
                question_type = f'yes_no_{attr}'

            elif 'open_end' in title_lower:
                # Visual_open_end_feature_1, Visual_open_end_color_2, ...
                m = re.search(
                    r'open_end_(\w+?)_(\d+)$', title, re.IGNORECASE
                )
                if not m:
                    continue
                attr          = m.group(1)
                obj_idx       = m.group(2)
                category      = f'1_Visual_open_end_{attr}'
                question_type = f'open_end_{attr}'

            elif 'external' in title_lower:
                # External_0, ExternalKnowledge_1, ...
                idx           = re.search(r'\d+', title)
                obj_idx       = idx.group() if idx else '0'
                category      = '2_ExternalKnowledge'
                question_type = 'ExternalKnowledge'

            else:
                continue  # title không nhận dạng được → bỏ qua

            if not question:
                continue

            qna.append({
                'category'     : category,
                'question_type': question_type,
                'question'     : question,
                'answer'       : answer,
                'sample_id'    : f'{image_id}_{question_type}_{obj_idx}'
            })

        results.append({
            'image_id'  : image_id,
            'image_path': f'images/image_{image_id}.jpg',
            'qna'       : qna
        })

    return results

In [ ]:
# ============================================================
# Cell #6 — Hàm gọi API: 1 batch ảnh + PROMPT / 1 request
# ============================================================

class DailyQuotaExceeded(Exception):
    pass

def call_api_batch(client, model_name, prompt, batch_ids, batch_paths,
                   max_retries=3):
    """
    Gọi API cho 1 batch ảnh.
    batch_ids   : list image_id thực tế
    batch_paths : list đường dẫn ảnh tương ứng
    Trả về: list dict đã parse, hoặc None nếu thất bại
    """
    n = len(batch_paths)

    # Build contents: [ảnh_1, ..., ảnh_n, prompt]
    contents = []
    for path in batch_paths:
        contents.append(types.Part.from_bytes(
            data=image_to_bytes(path), mime_type='image/jpeg'
        ))

    batch_prompt = (
        f'Bạn nhận được {n} ảnh đánh số image_1 đến image_{n}.\n\n'
        f'{prompt}'
    )
    contents.append(batch_prompt)

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=contents,
                config=types.GenerateContentConfig(
                    max_output_tokens=65536,
                    temperature=0.2
                )
            )
            raw        = response.text.strip()
            output_tok = response.usage_metadata.candidates_token_count

            # Cảnh báo nếu gần đầy token
            if output_tok >= 15000:
                print(f'    ⚠️  Output {output_tok}/65536 tokens — có thể bị cắt')

            # Parse text → list dict
            parsed = parse_text_response(raw, batch_ids)
            if len(parsed) == 0:
                raise ValueError('Parse ra 0 kết quả.')
            if len(parsed) < n:
                print(f'    ⚠️  Parse {len(parsed)}/{n} ảnh '
                      f'(output bị cắt {n-len(parsed)} ảnh cuối)')
                print(f'    [DEBUG] Tổng QA parsed: {sum(len(r["qna"]) for r in parsed)}')
                print(f'    [DEBUG] Sample title đầu: {raw[:200]}')  # xem raw response
            return parsed

        except DailyQuotaExceeded:
            raise
        except Exception as e:
            err = str(e).lower()
            print(f'    [Lỗi raw] attempt={attempt+1} | {str(e)[:120]}')

            daily_kw = ['quota exceeded', 'resource_exhausted',
                        'daily limit', 'per_day']
            rpm_kw   = ['429', 'rate limit', 'too many requests',
                        'per minute', 'per_minute']

            if any(w in err for w in daily_kw):
                print(f'    → DAILY QUOTA hết model {model_name}')
                raise DailyQuotaExceeded()
            elif any(w in err for w in rpm_kw):
                wait = 30 * (attempt + 1)
                print(f'    → RPM limit → đợi {wait}s...')
                time.sleep(wait)
            else:
                wait = (attempt + 1) * 5
                print(f'    → Lỗi khác → đợi {wait}s...')
                time.sleep(wait)

    print(f'    Bỏ qua batch ids={batch_ids[0]}→{batch_ids[-1]}')
    return None

print('✅ Cell #6 call_api_batch() ready.')

✅ Cell #6 call_api_batch() ready.


In [ ]:
# ============================================================
# Cell #7 — Worker: chạy song song nhiều API key
# Mỗi worker nhận: 1 account + list parts được gán
# Mỗi part = list batch, mỗi batch = list image_id
# ============================================================

_print_lock = threading.Lock()
def safe_print(msg):
    with _print_lock:
        print(msg)

def worker(account, assigned_parts, prompt, image_dir, output_dir):
    """
    account       : {'key', 'models', 'name'}
    assigned_parts: list of parts, mỗi part = {'model', 'batches': [[id,...], ...]}
    """
    acc_name = account['name']
    client   = genai.Client(api_key=account['key'])
    safe_print(f'[{acc_name}] Khởi tạo OK | '
               f'{len(assigned_parts)} parts | '
               f'{sum(len(p["batches"]) for p in assigned_parts)} batches')

    all_parsed   = []
    ckpt_file    = os.path.join(output_dir, f'checkpoint_{acc_name}.json')
    output_file  = os.path.join(output_dir, f'VN_Food_QAs_{acc_name}.json')

    # Load checkpoint
    if os.path.exists(ckpt_file):
        with open(ckpt_file, 'r', encoding='utf-8') as f:
            all_parsed = json.load(f)
        done_ids = {r['image_id'] for r in all_parsed}
        safe_print(f'[{acc_name}] Resume: {len(done_ids)} ảnh đã xong')
    else:
        done_ids = set()

    for part in assigned_parts:
        model_name = part['model']
        batches    = part['batches']
        part_id    = part['part_id']
        safe_print(f'[{acc_name}] ▶ Part {part_id} | model={model_name} | '
                   f'{len(batches)} batches ({len(batches)*BATCH_IMAGES} ảnh)')

        quota_exceeded = False
        for b_idx, batch_ids in enumerate(batches):
            # Lọc ảnh chưa xử lý
            pending_ids   = [iid for iid in batch_ids
                             if str(iid) not in done_ids]
            if not pending_ids:
                safe_print(f'  [{acc_name}] Batch {b_idx+1}/{len(batches)} '
                           f'— đã xong, bỏ qua')
                continue

            pending_paths = [os.path.join(image_dir, f'image_{iid}.jpg')
                             for iid in pending_ids]
            # Kiểm tra file tồn tại
            valid = [(iid, p) for iid, p in
                     zip(pending_ids, pending_paths)
                     if os.path.exists(p)]
            if not valid:
                safe_print(f'  [{acc_name}] Batch {b_idx+1}: '
                           f'không tìm thấy ảnh nào, bỏ qua')
                continue

            v_ids   = [str(v[0]) for v in valid]
            v_paths = [v[1]      for v in valid]

            safe_print(f'  [{acc_name}] Batch {b_idx+1}/{len(batches)} '
                       f'| {len(v_ids)} ảnh | ids {v_ids[0]}→{v_ids[-1]}')

            try:
                parsed = call_api_batch(
                    client, model_name, prompt, v_ids, v_paths
                )
            except DailyQuotaExceeded:
                safe_print(f'  [{acc_name}] Hết quota model {model_name} '
                           f'→ sang model tiếp theo')
                quota_exceeded = True
                break  # thoát khỏi vòng batch, sang part tiếp

            if parsed:
                all_parsed.extend(parsed)
                for r in parsed:
                    done_ids.add(r['image_id'])
                safe_print(f'  [{acc_name}] ✔ Batch {b_idx+1} xong | '
                           f'Tổng: {len(done_ids)} ảnh')

            time.sleep(SLEEP_SEC)

            # Checkpoint sau mỗi batch
            with open(ckpt_file, 'w', encoding='utf-8') as f:
                json.dump(all_parsed, f, ensure_ascii=False)

        # Nghỉ giữa các part
        if not quota_exceeded:
            safe_print(f'[{acc_name}] Part {part_id} xong | nghỉ {BATCH_SLEEP}s...')
            time.sleep(BATCH_SLEEP)

    # Lưu output cuối
    final = format_final_dataset(
        all_parsed, dataset_name=f'VN_Food_VQA_{acc_name}'
    )
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(final, f, ensure_ascii=False, indent=2)
    if os.path.exists(ckpt_file):
        os.remove(ckpt_file)

    safe_print(f'[{acc_name}] ✅ Xong! '
               f'{final["metadata"]["total_images"]} ảnh | '
               f'{final["metadata"]["total_QAs"]} QA → {output_file}')
    return all_parsed

print('✅ Cell #7 worker() ready.')

✅ Cell #7 worker() ready.


In [ ]:
# ============================================================
# Cell #8 — Chia ảnh → batch → part
#
# Cấu trúc:
#   all_ids         : [0, 1, 2, ..., 4999]  (đã sort)
#   batches         : [[0..14], [15..29], ..., [4995..4999]]  (batch cuối có thể < 15)
#   parts           : mỗi part = MAX_REQ_MODEL batch liên tiếp
#   part['part_id'] : 'part_acc{n}_model_{name}_p{idx}'
# ============================================================

# --- Đọc danh sách image_id từ thư mục (đã đổi tên 0..N) ---
raw_fnames = sorted(
    [f for f in os.listdir(IMAGE_DIR)
     if re.match(r'^image_\d+\.jpg$', f)],
    key=lambda x: int(re.search(r'\d+', x).group())
)
all_ids = [re.search(r'\d+', f).group() for f in raw_fnames]
print(f'Tổng ảnh trong thư mục: {len(all_ids)}')
print(f'id đầu: {all_ids[0]} | id cuối: {all_ids[-1]}')

# --- Giới hạn theo MODE ---
total_capacity = sum(
    len(acc['models']) * MAX_REQ_MODEL * BATCH_IMAGES
    for acc in ACCOUNTS
)
if MODE == 'demo':
    # Demo: chỉ lấy 1 batch để test
    run_ids = all_ids[:BATCH_IMAGES]
    print(f'[DEMO] Lấy {len(run_ids)} ảnh (1 batch)')
else:
    # Full: lấy tối đa theo capacity
    run_ids = all_ids[:total_capacity]
    print(f'[FULL] Lấy {len(run_ids)}/{len(all_ids)} ảnh '
          f'(capacity = {total_capacity})')

# --- Chia thành batches ---
# Các batch đều có BATCH_IMAGES ảnh, batch cuối chứa phần lẻ
batches = []
for i in range(0, len(run_ids), BATCH_IMAGES):
    batches.append(run_ids[i:i + BATCH_IMAGES])

# Kiểm tra batch cuối
last_batch_size = len(batches[-1]) if batches else 0
print(f'\nTổng batches     : {len(batches)}')
print(f'Batch thường     : {BATCH_IMAGES} ảnh')
print(f'Batch cuối       : {last_batch_size} ảnh'
      f'{" (lẻ)" if last_batch_size < BATCH_IMAGES else " (đủ)"}')

# --- Chia batches thành parts ---
# Mỗi part = MAX_REQ_MODEL batch liên tiếp
# part_id có dạng: part_{idx}_model_{model_name}
# (gán model sau ở Cell #9)
parts = []
for i in range(0, len(batches), MAX_REQ_MODEL):
    parts.append({
        'part_idx': len(parts),
        'batches' : batches[i:i + MAX_REQ_MODEL]
    })

print(f'\nTổng parts       : {len(parts)}')
print(f'Batches/part     : {MAX_REQ_MODEL} (= {MAX_REQ_MODEL * BATCH_IMAGES} ảnh/part)')
print(f'\nChi tiết 3 part đầu:')
for p in parts[:3]:
    ids_flat = [iid for b in p['batches'] for iid in b]
    print(f'  part_{p["part_idx"]:03d}: '
          f'{len(p["batches"])} batches | '
          f'{len(ids_flat)} ảnh | '
          f'id {ids_flat[0]}→{ids_flat[-1]}')

print('\n✅ Cell #8 xong.')

Tổng ảnh trong thư mục: 5001
id đầu: 0 | id cuối: 5000
[FULL] Lấy 5001/5001 ảnh (capacity = 6000)

Tổng batches     : 334
Batch thường     : 15 ảnh
Batch cuối       : 6 ảnh (lẻ)

Tổng parts       : 17
Batches/part     : 20 (= 300 ảnh/part)

Chi tiết 3 part đầu:
  part_000: 20 batches | 300 ảnh | id 0→299
  part_001: 20 batches | 300 ảnh | id 300→599
  part_002: 20 batches | 300 ảnh | id 600→899

✅ Cell #8 xong.


In [ ]:
# ============================================================
# Cell #9 — Gán parts cho từng tài khoản + model
#
# Logic:
#   Mỗi tài khoản được gán N parts liên tiếp
#   N = số model của tài khoản đó (mỗi model xử lý 1 part = 20 batch)
#   Gán theo thứ tự model ưu tiên: model tốt nhất nhận batch đầu tiên
#
# Ví dụ 10 tài khoản × 2 model × 20 batch = 400 batch:
#   acc_Key_1 → model_2.5-flash  : part_000 (batch 0-19)
#   acc_Key_1 → model_2.5-lite   : part_001 (batch 20-39)
#   acc_Key_2 → model_2.5-flash  : part_002 (batch 40-59)
#   ...
# ============================================================

part_cursor = 0  # index part tiếp theo chưa được gán

# assigned_work: list of (account, [parts...])
assigned_work = []

for acc in ACCOUNTS:
    acc_parts = []
    for model_name in acc['models']:
        if part_cursor >= len(parts):
            break  # hết parts
        part = parts[part_cursor].copy()
        part['model']   = model_name
        part['part_id'] = (f'part_{part["part_idx"]:03d}_'
                           f'{acc["name"]}_'
                           f'{model_name.replace("-","_")}')
        acc_parts.append(part)
        part_cursor += 1

    if acc_parts:
        assigned_work.append((acc, acc_parts))

# --- In phân công ---
print(f'Tổng parts đã gán: {part_cursor}/{len(parts)}')
if part_cursor < len(parts):
    remain = len(parts) - part_cursor
    print(f'⚠️  Còn {remain} parts chưa được gán '
          f'(cần thêm tài khoản hoặc chạy nhiều ngày)')

print(f'\nPhân công chi tiết:')
for acc, acc_parts in assigned_work:
    for p in acc_parts:
        ids_flat = [iid for b in p['batches'] for iid in b]
        print(f'  {p["part_id"]}')
        print(f'    → {len(p["batches"])} batches | '
              f'{len(ids_flat)} ảnh | '
              f'id {ids_flat[0]}→{ids_flat[-1]}')

print('\n✅ Cell #9 xong.')

Tổng parts đã gán: 17/17

Phân công chi tiết:
  part_000_vanh1_gemini_2.5_flash
    → 20 batches | 300 ảnh | id 0→299
  part_001_vanh1_gemini_2.5_flash_lite
    → 20 batches | 300 ảnh | id 300→599
  part_002_vanh2_gemini_2.5_flash
    → 20 batches | 300 ảnh | id 600→899
  part_003_vanh2_gemini_2.5_flash_lite
    → 20 batches | 300 ảnh | id 900→1199
  part_004_vanh3_gemini_2.5_flash
    → 20 batches | 300 ảnh | id 1200→1499
  part_005_vanh3_gemini_2.5_flash_lite
    → 20 batches | 300 ảnh | id 1500→1799
  part_006_vanh4_gemini_2.5_flash
    → 20 batches | 300 ảnh | id 1800→2099
  part_007_vanh4_gemini_2.5_flash_lite
    → 20 batches | 300 ảnh | id 2100→2399
  part_008_vanh5_gemini_2.5_flash
    → 20 batches | 300 ảnh | id 2400→2699
  part_009_vanh5_gemini_2.5_flash_lite
    → 20 batches | 300 ảnh | id 2700→2999
  part_010_vanh6_gemini_2.5_flash
    → 20 batches | 300 ảnh | id 3000→3299
  part_011_vanh6_gemini_2.5_flash_lite
    → 20 batches | 300 ảnh | id 3300→3599
  part_012_vanh7_gemi

In [ ]:
# ============================================================
# Cell #10 — DEMO: chạy 1 batch trên 1 tài khoản để test
# ============================================================

print('=' * 55)
print('DEMO: 1 batch, 1 tài khoản')
print('=' * 55)

# Lấy batch đầu tiên từ part đầu tiên của tài khoản đầu tiên
demo_account = assigned_work[0][0]
demo_part    = assigned_work[0][1][0]
demo_batch   = demo_part['batches'][0]
demo_model   = demo_part['model']

print(f'Tài khoản : {demo_account["name"]}')
print(f'Model     : {demo_model}')
print(f'Batch     : {len(demo_batch)} ảnh | ids {demo_batch[0]}→{demo_batch[-1]}')

# Build paths
demo_paths = [os.path.join(IMAGE_DIR, f'image_{iid}.jpg')
              for iid in demo_batch]
valid_demo = [(iid, p) for iid, p in zip(demo_batch, demo_paths)
              if os.path.exists(p)]
demo_ids   = [str(v[0]) for v in valid_demo]
demo_paths = [v[1]      for v in valid_demo]
print(f'Ảnh hợp lệ: {len(demo_ids)}/{len(demo_batch)}')

# Gọi API
client = genai.Client(api_key=demo_account['key'])
print(f'\n⏳ Gọi API...')
t0 = time.time()

parsed = call_api_batch(
    client, demo_model, PROMPT, demo_ids, demo_paths
)

elapsed = time.time() - t0
print(f'\n===== KẾT QUẢ DEMO =====')
print(f'Thời gian    : {elapsed:.1f}s')
print(f'Ảnh parse    : {len(parsed) if parsed else 0}/{len(demo_ids)}')

if parsed:
    total_qa = sum(len(r['qna']) for r in parsed)
    print(f'Tổng Q&A     : {total_qa}')
    print(f'Q&A/ảnh avg  : {total_qa/len(parsed):.1f}')

    # Format và lưu
    final_demo = format_final_dataset(parsed, 'VN_Food_VQA_DEMO')
    demo_file  = os.path.join(OUTPUT_DIR, 'demo_result.json')
    with open(demo_file, 'w', encoding='utf-8') as f:
        json.dump(final_demo, f, ensure_ascii=False, indent=2)
    print(f'\nFile lưu : {demo_file}')

    # Preview 2 Q&A đầu
    print(f'\nPreview 2 Q&A đầu:')
    for row in final_demo['data'][:2]:
        print(f'  [{row["question_type"]}] {row["question"]}')
        print(f'  → {row["answer"]}')
        print(f'  sample_id: {row["sample_id"]}')
else:
    print('❌ Không parse được kết quả nào — kiểm tra lại key/model')

print('\n✅ Cell #10 Demo xong.')

DEMO: 1 batch, 1 tài khoản
Tài khoản : vanh1
Model     : gemini-2.5-flash
Batch     : 15 ảnh | ids 0→14
Ảnh hợp lệ: 15/15

⏳ Gọi API...
    ⚠️  Output 19497/65536 tokens — có thể bị cắt

===== KẾT QUẢ DEMO =====
Thời gian    : 90.3s
Ảnh parse    : 30/15
Tổng Q&A     : 409
Q&A/ảnh avg  : 13.6

File lưu : /content/drive/MyDrive/Data_DL1/Output/demo_result.json

Preview 2 Q&A đầu:
  [Overall] Có gì trong bức ảnh này?
  → Bức ảnh này có món ăn là Bánh Bèo.
  sample_id: 0_Overall_0
  [Overall] Món ăn chính trong bức ảnh là gì?
  → Món ăn chính trong bức ảnh là Bánh Bèo.
  sample_id: 0_Overall_1

✅ Cell #10 Demo xong.


In [ ]:
# ============================================================
# Cell #11 — FULL: chạy song song nhiều tài khoản
# Chỉ chạy khi MODE = 'full' và đã test demo thành công
# ============================================================

if MODE != 'full':
    print('⚠️  MODE hiện tại là "demo" — đổi MODE="full" ở Cell #3 để chạy full.')
else:
    print('=' * 55)
    print(f'FULL RUN: {len(assigned_work)} tài khoản song song')
    print('=' * 55)

    all_results = []
    t_start     = time.time()

    with ThreadPoolExecutor(max_workers=len(assigned_work)) as executor:
        futures = {
            executor.submit(
                worker,
                acc,
                acc_parts,
                PROMPT,
                IMAGE_DIR,
                OUTPUT_DIR
            ): acc['name']
            for acc, acc_parts in assigned_work
        }

        for future in as_completed(futures):
            acc_name = futures[future]
            try:
                result = future.result()
                all_results.extend(result)
                print(f'[{acc_name}] Hoàn thành: {len(result)} ảnh')
            except Exception as e:
                print(f'[{acc_name}] LỖI: {e}')

    elapsed = time.time() - t_start
    print(f'\nTất cả worker xong trong {elapsed/60:.1f} phút')
    print(f'Tổng raw results: {len(all_results)} ảnh')

    # --- Merge tất cả → 1 file FINAL ---
    # Load thêm từ file worker nếu cần
    if len(all_results) == 0:
        for wf in sorted(glob.glob(
                os.path.join(OUTPUT_DIR, 'VN_Food_QAs_Key_*.json'))):
            with open(wf) as f:
                d = json.load(f)
            # Chuyển từ flat data về parsed format
            img_map = {}
            for row in d['data']:
                iid = row['image_id']
                if iid not in img_map:
                    img_map[iid] = {
                        'image_id'  : iid,
                        'image_path': row['image_path'],
                        'qna'       : []
                    }
                img_map[iid]['qna'].append({
                    'category'     : '',
                    'question_type': row['question_type'],
                    'question'     : row['question'],
                    'answer'       : row['answer'],
                    'sample_id'    : row['sample_id']
                })
            all_results.extend(img_map.values())
        print(f'Loaded từ file worker: {len(all_results)} ảnh')

    FINAL_OUTPUT  = os.path.join(OUTPUT_DIR, 'VN_Food_QAs_FINAL.json')
    final_dataset = format_final_dataset(all_results)
    with open(FINAL_OUTPUT, 'w', encoding='utf-8') as f:
        json.dump(final_dataset, f, ensure_ascii=False, indent=2)

    print(f'\n{"="*50}')
    print(f'✅ HOÀN TẤT')
    print(f'   Ảnh     : {final_dataset["metadata"]["total_images"]}')
    print(f'   QAs      : {final_dataset["metadata"]["total_QAs"]}')
    print(f'   File    : {FINAL_OUTPUT}')
    print(f'{"="*50}')

FULL RUN: 9 tài khoản song song
[vanh8] Khởi tạo OK | 2 parts | 40 batches
[vanh8] ▶ Part part_014_vanh8_gemini_2.5_flash | model=gemini-2.5-flash | 20 batches (300 ảnh)
[vanh5] Khởi tạo OK | 2 parts | 40 batches
[vanh4] Khởi tạo OK | 2 parts | 40 batches
[vanh1] Khởi tạo OK | 2 parts | 40 batches
[vanh2] Khởi tạo OK | 2 parts | 40 batches
[vanh4] ▶ Part part_006_vanh4_gemini_2.5_flash | model=gemini-2.5-flash | 20 batches (300 ảnh)
[vanh5] ▶ Part part_008_vanh5_gemini_2.5_flash | model=gemini-2.5-flash | 20 batches (300 ảnh)
[vanh1] ▶ Part part_000_vanh1_gemini_2.5_flash | model=gemini-2.5-flash | 20 batches (300 ảnh)
  [vanh8] Batch 1/20 | 15 ảnh | ids 4200→4214
[vanh2] ▶ Part part_002_vanh2_gemini_2.5_flash | model=gemini-2.5-flash | 20 batches (300 ảnh)
[vanh3] Khởi tạo OK | 2 parts | 40 batches
[vanh3] ▶ Part part_004_vanh3_gemini_2.5_flash | model=gemini-2.5-flash | 20 batches (300 ảnh)
  [vanh5] Batch 1/20 | 15 ảnh | ids 2400→2414
  [vanh1] Batch 1/20 | 15 ảnh | ids 0→14
  [vanh